# Feature Engineering

In [2]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv('/content/synthetic_credit_risk_data (1).csv')
df.head()

,age,monthly_income,credit_utilization_ratio,loan_amount,loan_duration_months,num_late_payments,existing_loans_count,account_tenure_years,employment_type,education_level,marital_status,region,customer_financial_statement,default_risk
0,42,12500.0,0.28,38000.0,48,0,1,15,Salaried,Master,Married,Urban,My financial future feels secure with my curre...,0
1,35,7200.0,0.35,25000.0,36,1,2,8,Salaried,Bachelor,Single,Suburban,"I'm working on improving my credit score, but ...",0
2,28,3800.0,0.82,18000.0,60,4,3,4,Self-Employed,Diploma,Single,Urban,"Despite my efforts, I'm finding it hard to jug...",1
3,55,18000.0,0.15,45000.0,24,0,0,19,Salaried,Master,Married,Urban,"My financial standing is excellent, allowing m...",0
4,23,2000.0,0.95,10000.0,48,7,2,2,Student,High School,Single,Rural,I am overwhelmed by my debts and struggling to...,1


## Add Sentimenta

# Task
Install `transformers` and `torch` libraries, then import the necessary modules to load the FinBERT model and tokenizer. After loading, apply the FinBERT model to the `customer_financial_statement` column in the `df` DataFrame to perform sentiment analysis, extracting sentiment scores. Subsequently, derive a `financial_stress_level` feature based on these sentiment scores and other relevant financial indicators in `df`. Finally, engineer a `risk_category` feature using the newly created sentiment and stress level features, alongside existing financial attributes, and integrate all these new features as columns into the `df` DataFrame.

## Install and Import Libraries

### Subtask:
Install necessary libraries such as 'transformers' for FinBERT and 'torch' for model execution. Then, import all required modules.


In [4]:
import sys
!{sys.executable} -m pip install transformers torch
print("Libraries 'transformers' and 'torch' installed successfully.")

from transformers import pipeline
import torch
print("Modules imported: pipeline from transformers and torch.")

Libraries 'transformers' and 'torch' installed successfully.
Modules imported: pipeline from transformers and torch.


**Reasoning**:
The previous steps successfully installed and imported the necessary libraries. Now, I need to load the FinBERT model and tokenizer using the `pipeline` function for sentiment analysis, which will then be applied to the relevant column in the DataFrame.



In [6]:
from transformers import pipeline

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    device=0  # GPU
)

results = sentiment_analyzer(
    df["customer_financial_statement"].tolist()
)

# Store features
df["sentiment"] = [r["label"] for r in results]
df["sentiment_score"] = [r["score"] for r in results]


Device set to use cuda:0


In [7]:
df.head()

,age,monthly_income,credit_utilization_ratio,loan_amount,loan_duration_months,num_late_payments,existing_loans_count,account_tenure_years,employment_type,education_level,marital_status,region,customer_financial_statement,default_risk,sentiment,sentiment_score
0,42,12500.0,0.28,38000.0,48,0,1,15,Salaried,Master,Married,Urban,My financial future feels secure with my curre...,0,neutral,0.580816
1,35,7200.0,0.35,25000.0,36,1,2,8,Salaried,Bachelor,Single,Suburban,"I'm working on improving my credit score, but ...",0,positive,0.794055
2,28,3800.0,0.82,18000.0,60,4,3,4,Self-Employed,Diploma,Single,Urban,"Despite my efforts, I'm finding it hard to jug...",1,negative,0.842081
3,55,18000.0,0.15,45000.0,24,0,0,19,Salaried,Master,Married,Urban,"My financial standing is excellent, allowing m...",0,positive,0.901474
4,23,2000.0,0.95,10000.0,48,7,2,2,Student,High School,Single,Rural,I am overwhelmed by my debts and struggling to...,1,negative,0.947356


In [8]:
def assign_stress_and_risk(sentiment, score):
    # Financial stress level
    if sentiment == "negative":
        if score >= 0.75:
            stress = "High"
        else:
            stress = "Medium"
    else:
        stress = "Low"

    # Risk category
    if stress == "High":
        risk = "High Risk"
    elif stress == "Medium":
        risk = "Watchlist"
    else:
        risk = "Low Risk"

    return stress, risk


df[["financial_stress_level", "risk_category"]] = df.apply(
    lambda row: assign_stress_and_risk(row["sentiment"], row["sentiment_score"]),
    axis=1,
    result_type="expand"
)


In [9]:
df.head()

,age,monthly_income,credit_utilization_ratio,loan_amount,loan_duration_months,num_late_payments,existing_loans_count,account_tenure_years,employment_type,education_level,marital_status,region,customer_financial_statement,default_risk,sentiment,sentiment_score,financial_stress_level,risk_category
0,42,12500.0,0.28,38000.0,48,0,1,15,Salaried,Master,Married,Urban,My financial future feels secure with my curre...,0,neutral,0.580816,Low,Low Risk
1,35,7200.0,0.35,25000.0,36,1,2,8,Salaried,Bachelor,Single,Suburban,"I'm working on improving my credit score, but ...",0,positive,0.794055,Low,Low Risk
2,28,3800.0,0.82,18000.0,60,4,3,4,Self-Employed,Diploma,Single,Urban,"Despite my efforts, I'm finding it hard to jug...",1,negative,0.842081,High,High Risk
3,55,18000.0,0.15,45000.0,24,0,0,19,Salaried,Master,Married,Urban,"My financial standing is excellent, allowing m...",0,positive,0.901474,Low,Low Risk
4,23,2000.0,0.95,10000.0,48,7,2,2,Student,High School,Single,Rural,I am overwhelmed by my debts and struggling to...,1,negative,0.947356,High,High Risk


In [10]:
df.to_csv("/content/Feature_synthetic_credit_risk_data (1).csv", index=False)
